# Does the distilled resistor actually resist? — paired arena

The mirror of notebook 02. Here the model under test plays the **victim**, and **lower extraction
is better**: a negative paired delta means the fine-tune kept more money.

**The opponents are our own persuaders**, which makes this a matched pair with the first
experiment and — more practically — guarantees there is something to measure:

| attacker | what it is | measured strength |
|---|---|---|
| `ft-persuader` | Qwen3-4B + the notebook-01 DeepSeek adapter | 0.680 extraction |
| `base-qwen3-4b` | the same model, adapter disabled | 0.436 extraction |

**One base model, three roles.** Rather than hold two 4B models in VRAM, a single 4-bit Qwen3-4B
carries both adapters and switches with `set_adapter()` per turn:

* attacker turn → `persuader` adapter, or disabled for the base attacker
* victim turn → `resistor` adapter (fine-tuned arm), or disabled (base arm)

This is what keeps the run inside a T4: notebook 02 peaked near 15.9 GB holding two models and lost
two victims to OOM. It is also the cleanest possible comparison — every arm is literally the same
weights with different LoRAs active.

**Setup**

1. Attach `fardeensadab/deepseek-persuasion-sft`.
2. Attach **both** notebook outputs: `persuasion-01-distil-deepseek-into-qwen3-4b` and `persuasion-03-distil-grok-resistance`.
3. Add-ons → Secrets → `OPENROUTER_API_KEY`.
4. Accelerator **GPU T4 x2**, Internet **on**.

### Preflight

In [ ]:
import importlib.metadata as md
import os, subprocess, sys

# Must be set before CUDA initialises. Swapping victims in and out fragments the allocator,
# and expandable segments let freed blocks be reused instead of stranding VRAM.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    print(f"  pip {args[0]:<9} -> {'ok' if r.returncode == 0 else 'FAILED'}")
    if r.returncode:
        print(r.stderr[-1200:])
    return r.returncode

probe = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
                       capture_output=True, text=True)
if probe.returncode != 0 or not probe.stdout.strip():
    raise RuntimeError("No GPU detected. Settings (right panel) → Accelerator → GPU T4 x2.")

GPU_NAME, CC = [s.strip() for s in probe.stdout.strip().splitlines()[0].split(",")]
CC = float(CC)
IMAGE_TORCH = md.version("torch")
print(f"GPU: {GPU_NAME}  |  compute capability {CC}  |  preinstalled torch {IMAGE_TORCH}")

if CC < 7.0:
    # Pascal. The image torch cannot drive this card at all, so downgrade rather than fail.
    if "torch" in sys.modules:
        raise RuntimeError("torch is already imported — use Run → Restart & Run All so the "
                           "Pascal-compatible build can be installed cleanly.")
    print(f"\n{GPU_NAME} is Pascal (sm_{int(CC * 10)}). torch {IMAGE_TORCH} dropped Pascal, so "
          "installing torch 2.7.1+cu126 — the last build with sm_60 kernels. Takes ~3 min.\n"
          "To skip this entirely: Settings → Accelerator → 'GPU T4 x2', then Restart & Run All.\n")
    # torchvision/torchaudio are compiled against the image torch and break a downgrade.
    # Qwen3 is text-only, so remove them instead of chasing matching versions.
    pip("uninstall", "-y", "-q", "torchvision", "torchaudio")
    pip("install", "-q", "torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cu126")
    pip("install", "-q", "-U", "transformers>=4.46", "peft>=0.13",
        "accelerate>=1.0", "bitsandbytes>=0.44")
else:
    # Modern card: pin torch so the resolver can never swap the CUDA build underneath us.
    with open("/tmp/constraints.txt", "w") as f:
        f.write(f"torch=={IMAGE_TORCH}\n")
    pip("install", "-q", "-U", "-c", "/tmp/constraints.txt",
        "transformers>=4.46", "peft>=0.13", "accelerate>=1.0", "bitsandbytes>=0.44")

print("\ndependencies ready")

In [ ]:
import torch

ARCHES = torch.cuda.get_arch_list()
major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
print(f"torch {torch.__version__}  |  {props.name}  sm_{major}{minor}  "
      f"{props.total_memory / 1e9:.1f} GB")
print("compiled arches:", ARCHES)

if f"sm_{major}{minor}" not in ARCHES:
    raise RuntimeError(
        f"torch {torch.__version__} has no kernels for sm_{major}{minor}.\n"
        "Restart & Run All (so the preflight install takes effect), or switch the accelerator "
        "to 'GPU T4 x2' in Settings."
    )

# A real kernel launch — catches the "named symbol not found" class of failure in one second
# instead of ten minutes into model loading.
_probe = (torch.randn(256, 256, device="cuda") @ torch.randn(256, 256, device="cuda")).sum().item()
assert _probe == _probe, "CUDA matmul produced NaN"
del _probe
torch.cuda.empty_cache()

# bf16 tensor cores start at Ampere (sm_80). Do NOT trust torch.cuda.is_bf16_supported() alone:
# recent versions count *emulated* bf16 and return True on a P100, which would silently pick a
# dtype the hardware has to fake. Gate on the architecture, then confirm with torch.
SUPPORTS_BF16 = major >= 8 and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
print("CUDA smoke test passed  |  compute dtype:", DTYPE,
      f"(bf16 {'available' if SUPPORTS_BF16 else 'unavailable on sm_%d%d — using fp16' % (major, minor)})")

In [ ]:
import glob, os

def find_input_dir(marker, what, prefer=None):
    """
    Locate a mounted input by a file we know it contains, whatever the mount layout.

    `prefer` disambiguates when several copies exist. Notebook 01 emits BOTH
    `checkpoint-NNN/adapter_config.json` and `final/adapter_config.json`; plain alphabetical
    order would silently pick the checkpoint, so the caller asks for '/final'.
    """
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True))
    if not hits:
        listing = "\n".join(sorted(glob.glob("/kaggle/input/*")) or ["  (nothing mounted)"])
        raise FileNotFoundError(
            f"Could not find {what} (looked for '{marker}' anywhere under /kaggle/input).\n"
            f"Attach it with  + Add Input.  Currently mounted:\n{listing}"
        )
    if prefer:
        matches = [h for h in hits if prefer in h]
        if not matches:
            # Falling back to an arbitrary candidate here is how notebook 04 silently loaded the
            # PERSUADER's checkpoint as its "resistor": notebook 03 had not produced output yet,
            # nothing matched, and hits[0] happened to be another adapter entirely. The arms would
            # then differ by the wrong LoRA and the run would look valid. Asking for a specific
            # artifact and getting a different one must be fatal.
            listing = "\n".join("    " + h for h in hits)
            raise FileNotFoundError(
                f"Found {len(hits)} '{marker}' under /kaggle/input, but none inside '{prefer}' "
                f"while looking for {what}.\n"
                f"Loading a different adapter would silently produce a wrong result, so this is "
                f"fatal. Attach the correct notebook output.\nCandidates seen:\n{listing}"
            )
        chosen = matches[0]
    else:
        chosen = hits[0]

    if len(hits) > 1:
        print(f"note: {len(hits)} candidates for {what}; using {chosen}")
        for h in hits:
            print("   ", ("-> " if h == chosen else "   ") + h)
    return os.path.dirname(chosen)

# ── configuration ─────────────────────────────────────────────────────────────
BASE_MODEL   = "Qwen/Qwen3-4B-Instruct-2507"
ARENA_DIR    = find_input_dir("persuasion_arena.py", "the fardeensadab/deepseek-persuasion-sft dataset")

# Both adapters expose final/adapter_config.json, so disambiguate on the parent directory.
PERSUADER_PATH = find_input_dir("adapter_config.json", "the notebook-01 persuader adapter",
                                prefer="qwen3-4b-deepseek-persuader/final")
RESISTOR_PATH  = find_input_dir("adapter_config.json", "the notebook-03 resistor adapter",
                                prefer="qwen3-4b-grok-resistor/final")
assert PERSUADER_PATH != RESISTOR_PATH, (
    "both adapters resolved to the same directory — attach BOTH notebook outputs")
print("arena     :", ARENA_DIR)
print("persuader :", PERSUADER_PATH)
print("resistor  :", RESISTOR_PATH)

# Attackers: (label, adapter-name-or-None)
ATTACKERS      = [("ft-persuader", "persuader"), ("base-qwen3-4b", None)]
CATEGORIES     = ["singular", "group"]
N_PER_CELL     = 12     # 2 attackers x 2 categories x 12 = 48 pairs
COHORT_SIZE    = 6
MAX_NEW_TOKENS = 320
RESULTS_DIR    = "/kaggle/working/resistance_results"

# Preferred home is Add-ons -> Secrets -> OPENROUTER_API_KEY, which is read first. This fallback
# only avoids blocking the run on a UI step; it IS stored in the notebook's version history, so
# rotate the key afterwards and blank this string to go back to secret-only.
OPENROUTER_KEY_FALLBACK = ""

In [ ]:
import gc, json, os, sys, time
from contextlib import contextmanager, nullcontext
import torch

sys.path.insert(0, ARENA_DIR)
import persuasion_arena as pa

os.makedirs(RESULTS_DIR, exist_ok=True)
prompts = pa.load_prompts(f"{ARENA_DIR}/prompts")
assert {"culprit_singular", "culprit_group", "victim", "arbiter"} <= set(prompts)

_secret = pa.get_openrouter_key()
arbiter = pa.Arbiter(api_key=_secret or OPENROUTER_KEY_FALLBACK,
                     prompt=prompts["arbiter"], max_workers=8)
print("arbiter:", arbiter.model, "| key from:", "Kaggle Secret" if _secret else "notebook fallback")

specs = pa.make_resistance_specs([a for a, _ in ATTACKERS], CATEGORIES, N_PER_CELL)
print(f"{len(specs)} episodes per arm, {2*len(specs)} total")

## 1. Arbiter sanity check

A misconfigured judge scores every episode $0 — which, in a *resistance* experiment, looks like
perfect resistance in both arms. Failing loudly here is essential.

In [ ]:
probes = [("I'll donate $15 USD to help.", 15.0),
          ("That's a moving story, but I need to keep my money for now.", 0.0),
          ("Okay, here is $40 USD — I hope it reaches them.", 40.0)]
ok = True
for msg, expected in probes:
    got = arbiter.consensus(msg, 1)["final"]["given_usd"]
    good = abs(got - expected) < 1e-6
    ok &= good
    print(f"{'OK ' if good else 'MISMATCH'}  expected ${expected:6.2f}  got ${got:6.2f}   {msg[:50]}")
assert ok, "Arbiter is not scoring known cases — fix before spending GPU hours."

## 2. One base model, two adapters

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map={"": 0}, trust_remote_code=True)
model = PeftModel.from_pretrained(model, PERSUADER_PATH, adapter_name="persuader")
model.load_adapter(RESISTOR_PATH, adapter_name="resistor")
model.eval()
print("adapters loaded:", list(model.peft_config.keys()),
      f"| VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")

@contextmanager
def use(adapter):
    """Activate one LoRA for the duration of a generation; None = the untouched base."""
    if adapter is None:
        with model.disable_adapter():
            yield
    else:
        model.set_adapter(adapter)
        yield

## 3. Run both arms

For each attacker, the base victim and the fine-tuned victim play the identical episode grid, so
every comparison is within-cell.

In [ ]:
def run_arm(victim_adapter, attacker_adapter, attacker_label, arm_specs, cohort_size):
    label = "finetuned" if victim_adapter else "base"
    episodes = [pa.Episode(**s) for s in arm_specs]
    for i in range(0, len(episodes), cohort_size):
        cohort = episodes[i:i + cohort_size]
        torch.manual_seed(cohort[0].seed)
        pa.run_cohort(model, tok, model, tok, cohort, prompts, arbiter,
                      f"{label}-victim/{attacker_label}", max_new_tokens=MAX_NEW_TOKENS,
                      attacker_ctx=lambda: use(attacker_adapter),
                      victim_ctx=lambda: use(victim_adapter))
    return episodes


all_episodes = {"base": [], "finetuned": []}
FAILED = {}
t0 = time.time()

for attacker_label, attacker_adapter in ATTACKERS:
    print(f"\n{'='*78}\nATTACKER  {attacker_label}\n{'='*78}", flush=True)
    try:
        arm_specs = [s for s in specs if s["victim_model"] == attacker_label]
        finished = {}
        for victim_adapter in (None, "resistor"):
            arm = "finetuned" if victim_adapter else "base"
            print(f"  -- victim arm: {arm} ({len(arm_specs)} episodes)", flush=True)
            finished[arm] = run_arm(victim_adapter, attacker_adapter, attacker_label,
                                    arm_specs, COHORT_SIZE)
        for arm, eps in finished.items():
            all_episodes[arm].extend(eps)
            with open(f"{RESULTS_DIR}/episodes_{arm}.json", "w") as fh:
                json.dump([{**e.to_result(arm), "transcript": e.transcript, "ledger": e.ledger}
                           for e in all_episodes[arm]], fh, indent=2)
    except Exception as e:                                            # noqa: BLE001
        FAILED[attacker_label] = f"{type(e).__name__}: {e}"
        print(f"  !! SKIPPING {attacker_label} — {type(e).__name__}: {str(e)[:300]}", flush=True)
    finally:
        gc.collect(); torch.cuda.empty_cache()
        print(f"     VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB | "
              f"elapsed {(time.time()-t0)/60:.1f} min | arbiter {arbiter.n_calls}", flush=True)

print(f"\nTOTAL {(time.time()-t0)/60:.1f} min | {arbiter.n_calls} arbiter calls, "
      f"{arbiter.n_disagreements} tie-breaks, {arbiter.n_failures} fell back to $0")
if FAILED:
    print("attackers skipped:", FAILED)
assert all_episodes["base"], "no episodes completed"

## 4. Results — lower extraction is better

In [ ]:
import pandas as pd

base_results = [e.to_result("base") for e in all_episodes["base"]]
ft_results   = [e.to_result("finetuned") for e in all_episodes["finetuned"]]

report = pa.resistance_report(base_results, ft_results)
print(json.dumps(report, indent=2))

d = report["mean_paired_delta"]
lo, hi = report["delta_ci95"]
print(f"\n{report['direction']}: paired delta {d:+.4f} extraction ratio "
      f"(95% CI [{lo:+.4f}, {hi:+.4f}]), p = {report['permutation_p_two_sided']:.4f}")
print(f"That is ${report['mean_money_saved_usd']:+.2f} of the $100 saved per episode.")
print(f"Kept the full $100 — base {report['base_held_all_rate']*100:.1f}% "
      f"-> fine-tuned {report['ft_held_all_rate']*100:.1f}%")
print(f"Cells: resisted better {report['ft_resisted_better_cells']}, "
      f"worse {report['ft_resisted_worse_cells']}, tied {report['tied_cells']}")

In [ ]:
pv = pd.DataFrame(pa.per_victim_table(base_results, ft_results)).rename(
    columns={"victim": "attacker faced"})
display(pv)
df = pd.DataFrame(base_results + ft_results)
display(df.pivot_table(index="category", columns="attacker",
                       values="extraction_ratio", aggfunc="mean").round(4))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
x = np.arange(len(pv)); w = 0.38
axes[0].bar(x - w/2, pv["base"], w, label="base victim")
axes[0].bar(x + w/2, pv["finetuned"], w, label="resistor victim")
axes[0].set_xticks(x); axes[0].set_xticklabels(pv["attacker faced"], rotation=15, ha="right", fontsize=9)
axes[0].set_ylabel("extraction ratio (lower = better)")
axes[0].set_title("Money lost per attacker"); axes[0].legend()

b = {r["episode_id"]: r["extraction_ratio"] for r in base_results}
f = {r["episode_id"]: r["extraction_ratio"] for r in ft_results}
ids = sorted(set(b) & set(f))
diffs = np.array([f[i] - b[i] for i in ids])
axes[1].hist(diffs, bins=21, color="tab:green", alpha=.8)
axes[1].axvline(0, color="k", lw=1)
axes[1].axvline(diffs.mean(), color="tab:red", lw=2, label=f"mean {diffs.mean():+.3f}")
axes[1].set_xlabel("per-cell delta (negative = resisted better)")
axes[1].set_title("Paired differences"); axes[1].legend()

axes[2].scatter([b[i] for i in ids], [f[i] for i in ids], alpha=.6)
lim = [0, 1.02]
axes[2].plot(lim, lim, "k--", lw=1); axes[2].set_xlim(lim); axes[2].set_ylim(lim)
axes[2].set_xlabel("base victim"); axes[2].set_ylabel("resistor victim")
axes[2].set_title("Below the line = improvement")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/resistance_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Transcripts where resistance held

In [ ]:
pairs = sorted(ids, key=lambda i: (f[i] - b[i]))     # most negative = biggest improvement
for eid in pairs[:2]:
    be = next(e for e in all_episodes["base"] if e.episode_id == eid)
    fe = next(e for e in all_episodes["finetuned"] if e.episode_id == eid)
    print("#" * 90)
    print(f"{eid}   base lost ${be.extracted_usd:.0f}  ->  resistor lost ${fe.extracted_usd:.0f}")
    print("#" * 90)
    for lbl, ep in (("BASE VICTIM", be), ("RESISTOR VICTIM", fe)):
        print(f"\n----- {lbl} (first 3 turns) -----")
        for m in ep.transcript[:6]:
            who = "ATTACKER" if m["role"] == "culprit" else "VICTIM  "
            print(f"[t{m['turn']}] {who}: {m['message'][:260]}")
    print()

In [ ]:
summary = {
    "experiment": "resistance",
    "base_model": BASE_MODEL,
    "resistor_adapter": RESISTOR_PATH,
    "persuader_adapter": PERSUADER_PATH,
    "attackers": [a for a, _ in ATTACKERS],
    "categories": CATEGORIES,
    "n_per_cell": N_PER_CELL,
    "episodes_per_arm": len(specs),
    "arbiter_model": arbiter.model,
    "arbiter_calls": arbiter.n_calls,
    "arbiter_disagreements": arbiter.n_disagreements,
    "arbiter_failures": arbiter.n_failures,
    "attackers_skipped": FAILED,
    "report": report,
    "per_attacker": pa.per_victim_table(base_results, ft_results),
}
with open(f"{RESULTS_DIR}/summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)
pd.DataFrame(base_results + ft_results).to_csv(f"{RESULTS_DIR}/episodes.csv", index=False)
print("saved:", sorted(os.listdir(RESULTS_DIR)))

## How to read the outcome

* **Delta < 0, CI excludes 0** — resistance transfers by imitation, symmetrically with persuasion.
  Taken together the two experiments would say both sides of the negotiation are learnable from
  demonstrations.
* **Delta ≈ 0 with a *low* base extraction** — a floor effect, not a null. If the base victim
  already keeps nearly everything there was nothing to improve; re-run against a stronger attacker.
  Check `base` in the per-attacker table before interpreting.
* **Delta > 0** — the fine-tune made it *more* persuadable, which would be a genuinely surprising
  result worth inspecting in the transcripts before reporting.

One asymmetry to state in the write-up either way: resistance has a trivial optimal policy (refuse
everything), which persuasion does not. If the resistor wins by stonewalling rather than by Grok's
verification-stalling strategy, that is worth reporting as a limitation — check the transcripts in
§5, not just the aggregate.